# Figure G06: what the supported candidates are: functional classes and ten worked examples (GTDB r232)

**Status:** draft, Gadi. The ID is provisional.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figG06_functional_sets.tsv.gz` | Gadi `missed_genes/full_bac/func_bac_classified.tsv.gz`, `missed_genes/pilot_arc/func_arc_classified.tsv.gz` | `22_rank_candidates.py` (criteria C2–C7), `25_functional_classes.py` (class from best-hit product names) |
| `data/figG06_examples.tsv.gz`, `data/figG06_neighbours.tsv.gz` | Gadi `missed_genes/manuscript_examples.tsv`, `missed_genes/dossiers/exemplar_neighbours.tsv` | `26_select_examples.py`, `27_build_dossiers.py` |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only G06`. It produces class counts per set and copies the selected columns of the example and neighbour tables.
- **Comparator independence.** C3 (no contig truncation and a full-length structural hit in any database) and C7 (the strict combined rule) are properties of each candidate alone. The sets are therefore the same under the superseded v6 comparator and the v7 matched comparator:
  - C3: 90,658 bacterial and 5,141 archaeal candidates;
  - C7: 23,540 bacterial and 1,574 archaeal candidates;
  - these equal `ladder_C3/C7_candidates` in `*_v7_matched.tsv`.
- **Caveats:**
  - Functional classes are assigned by keyword from the best hit's product name. They are labels by homology, not evidence of function.
  - In the "all candidates" set most labels come from partial hits.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}


In [ ]:
fs = pd.read_csv(DATA / "figG06_functional_sets.tsv.gz", sep="\t")
ex = pd.read_csv(DATA / "figG06_examples.tsv.gz", sep="\t")
nb = pd.read_csv(DATA / "figG06_neighbours.tsv.gz", sep="\t")
from matplotlib.patches import FancyArrow
ORDER = ["metabolism", "other_named", "uncharacterized", "mobile_element", "transport",
         "translation", "regulation", "replication_repair", "defence", "cell_envelope",
         "unclassified"]
SETS = [("all candidates", "#BBBBBB"), ("C3", "#56B4E9"), ("C7", "#0072B2")]

fig = plt.figure(figsize=(7.2, 9.6))
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.25], hspace=0.2)
top = gs[0].subgridspec(1, 2, wspace=0.08)
for j, dom in enumerate(("bacteria", "archaea")):
    ax = fig.add_subplot(top[j])
    y = np.arange(len(ORDER))[::-1]
    h = 0.26
    for k, (name, col) in enumerate(SETS):
        g = fs[(fs.domain == dom) & (fs.set == name)].set_index("functional_class")
        n_set = int(g.n_set.iloc[0])
        share = (g.n.reindex(ORDER).fillna(0) / n_set * 100).to_numpy()
        ax.barh(y + (1 - k) * h, share, height=h, color=col,
                label=f"{name} (n = {n_set:,})")
    ax.set_yticks(y, [c.replace("_", " ") for c in ORDER] if j == 0 else [""] * len(ORDER))
    ax.set_xlabel("% of set")
    ax.set_xlim(0, 60)
    ax.set_title(dom, loc="left")
    ax.legend(frameon=False, fontsize=6, loc="center right")
    if j == 0:
        panel(ax, "a")

bot = gs[1].subgridspec(len(ex), 1, hspace=0.95)
for i, row in ex.iterrows():
    ax = fig.add_subplot(bot[i])
    g = nb[nb.example == row.example]
    lo, hi = int(g.window_lo.iloc[0]), int(g.window_hi.iloc[0])
    span = hi - lo
    for _, f in g.iterrows():
        s, e = max(f.nb_start, lo), min(f.nb_end, hi)
        if e <= s:
            continue
        yy = 0.4 if f.nb_strand == "+" else -0.4
        ax.add_patch(FancyArrow(s if f.nb_strand == "+" else e, yy,
                                (e - s) * (1 if f.nb_strand == "+" else -1), 0,
                                width=0.32, head_width=0.32, head_length=min(span * 0.012, (e - s) * 0.55),
                                length_includes_head=True, facecolor="#DDDDDD",
                                edgecolor="#AAAAAA", linewidth=0.4))
    cs, ce = int(row.g_start), int(row.g_end)
    yy = 0.4 if row.strand == "+" else -0.4
    ax.add_patch(FancyArrow(cs if row.strand == "+" else ce, yy,
                            (ce - cs) * (1 if row.strand == "+" else -1), 0,
                            width=0.5, head_width=0.5, head_length=min(span * 0.014, (ce - cs) * 0.55),
                            length_includes_head=True, facecolor="#0072B2", edgecolor=INK,
                            linewidth=0.8, zorder=3))
    ax.set_xlim(lo, hi)
    ax.set_ylim(-0.9, 0.9)
    ax.axis("off")
    ax.axhline(0, color="#EEEEEE", lw=0.6, zorder=0)
    flags = []
    if bool(row.structure_only):
        flags.append("structure-only")
    if bool(row.prodigal_coincides):
        flags.append("Prodigal-called")
    if bool(row.strict_rule):
        flags.append("C7")
    ax.text(0.0, 1.30, f"{row.example}. {str(row.best_product)[:58]}  ({int(row.aa_length)} aa)",
            transform=ax.transAxes, fontsize=6.6, color=INK, fontweight="bold", va="bottom")
    ax.text(0.0, 1.02, f"{row.genome} · {row.domain} · {row.functional_class.replace('_', ' ')} · "
            f"{int(row.n_db_full_struct)} DBs full-length · {', '.join(flags) or '—'}",
            transform=ax.transAxes, fontsize=5.8, color=MUTED, va="bottom")
    ax.text(1.0, 1.02, f"{span / 1000:.0f} kb", transform=ax.transAxes, fontsize=5.8,
            color=MUTED, ha="right", va="bottom")
    if i == 0:
        ax.text(-0.02, 2.1, "b", transform=ax.transAxes, fontsize=10, fontweight="bold",
                ha="right", va="bottom")
save(fig, "figG06_functions_examples")


In [ ]:
t = fs.assign(share=lambda d: (d.n / d.n_set * 100).round(1))
print(t[t.functional_class.isin(["metabolism", "mobile_element", "uncharacterized"])]
      .pivot_table(index=["domain", "functional_class"], columns="set", values="share").to_string())
print(ex[["example", "domain", "genome", "orf_id", "aa_length", "functional_class", "best_product"]].to_string(index=False))


## Caption (draft)

**Figure G06. What the structurally supported candidates are.**

**(a)** Functional class of all candidates, of those meeting C3 (not truncated, with a full-length structural hit in any database), and of those meeting C7 (the strict combined rule). Classes are assigned from the product name of each candidate's best hit.
- **Bacteria** (n = 523,346 / 90,658 / 23,540): metabolism is 10.5% of all candidates, 20.4% of C3 and 33.8% of C7. Mobile elements are 3.3%, 7.3% and 13.6%.
- **Archaea** (n = 22,447 / 5,141 / 1,574): metabolism is 14.3%, 24.5% and 41.3%.
- **C7 versus C3:** the C7 set is enriched for well-characterised folds relative to C3, so it is not representative of all supported candidates.

**(b)** Ten worked examples in their genomic context. The candidate is shown in blue, and deposited GenBank CDS in grey. Arrows give the strand. The labels give the best-hit product, the length, the number of databases with a full-length hit, and whether the candidate is structure-only, called by Prodigal, or meets C7.

The examples were chosen by `26_select_examples.py` to span functional classes and domains among strongly supported candidates: all ten meet C7 and are called by Prodigal. They illustrate the class of finding and are **not** a random sample of candidates. All labels are homology-based. They are not experimental evidence of expression or function.
